# Phase 2: Model Training & Evaluation

## Setup — Load Preprocessed Data from Phase 1

In [2]:
# ============================================================
# Install dependencies & restart runtime
# Run this cell, then go to Runtime > Restart session
# ============================================================
!pip install datasets==2.14.7 huggingface_hub==0.21.4 transformers==4.40.2 -q
!pip install underthesea scikit-learn -q
print("✅ Done. Now go to Runtime > Restart session, then run the next cell.")

✅ Done. Now go to Runtime > Restart session, then run the next cell.


In [1]:
# ============================================================
# Import libraries and mount Google Drive
# Load preprocessed CSVs saved at the end of Phase 1
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import json
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DATA_DIR = '/content/drive/MyDrive/phobert-sentiment-analysis/data'
train_df = pd.read_csv(f'{DATA_DIR}/train_preprocessed.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val_preprocessed.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test_preprocessed.csv')

print(f"Loaded — Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")
print(f"Columns: {list(train_df.columns)}")
train_df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Loaded — Train: 11,426, Val: 1,583, Test: 3,166
Columns: ['sentence', 'sentiment', 'topic', 'label_name', 'char_length', 'word_count', 'clean_text', 'clean_word_count', 'token_length']


,sentence,sentiment,topic,label_name,char_length,word_count,clean_text,clean_word_count,token_length
0,slide giáo trình đầy đủ .,2,1,Positive,25,6,slide giáo_trình đầy_đủ .,4,6
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0,Positive,46,11,"nhiệt_tình giảng_dạy , gần_gũi với sinh_viên .",7,9
2,đi học đầy đủ full điểm chuyên cần .,0,1,Negative,36,9,đi học đầy_đủ full_điểm chuyên cần .,7,11


In [3]:
# ============================================================
# Rebuild tokenizer, Dataset class, and DataLoaders
# Same config as Phase 1 — MAX_LENGTH=128, BATCH_SIZE=32
# ============================================================
LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']
NUM_CLASSES = 3
MAX_LENGTH = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

class VietnameseSentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe['clean_text'].tolist()
        self.labels = dataframe['sentiment'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = VietnameseSentimentDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = VietnameseSentimentDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = VietnameseSentimentDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready — Train: {len(train_loader)} batches, Val: {len(val_loader)}, Test: {len(test_loader)}")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataLoaders ready — Train: 358 batches, Val: 50, Test: 99


## 2.1 Baseline — TF-IDF + Logistic Regression

In [4]:
# ============================================================
# Baseline: TF-IDF + Logistic Regression
# Establishes a non-neural benchmark before fine-tuning PhoBERT
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_val_tfidf = tfidf.transform(val_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

lr_model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_model.fit(X_train_tfidf, train_df['sentiment'])

y_val_pred = lr_model.predict(X_val_tfidf)
y_test_pred = lr_model.predict(X_test_tfidf)

print("=== Baseline — Validation Set ===")
print(f"Accuracy: {accuracy_score(val_df['sentiment'], y_val_pred):.4f}")
print(classification_report(val_df['sentiment'], y_val_pred, target_names=LABEL_NAMES))

print("=== Baseline — Test Set ===")
print(f"Accuracy: {accuracy_score(test_df['sentiment'], y_test_pred):.4f}")
print(classification_report(test_df['sentiment'], y_test_pred, target_names=LABEL_NAMES))

=== Baseline — Validation Set ===
Accuracy: 0.9097
              precision    recall  f1-score   support

    Negative       0.88      0.96      0.92       705
     Neutral       0.86      0.08      0.15        73
    Positive       0.93      0.94      0.94       805

    accuracy                           0.91      1583
   macro avg       0.89      0.66      0.67      1583
weighted avg       0.91      0.91      0.89      1583

=== Baseline — Test Set ===
Accuracy: 0.8904
              precision    recall  f1-score   support

    Negative       0.86      0.97      0.91      1409
     Neutral       0.71      0.06      0.11       167
    Positive       0.93      0.91      0.92      1590

    accuracy                           0.89      3166
   macro avg       0.83      0.65      0.65      3166
weighted avg       0.88      0.89      0.87      3166



In [6]:
# ============================================================
# Store baseline results for later comparison (Phase 3)
# ============================================================
results = {
    'model': [],
    'val_accuracy': [],
    'test_accuracy': [],
    'test_f1_macro': []
}

results['model'].append('TF-IDF + LogReg')
results['val_accuracy'].append(accuracy_score(val_df['sentiment'], y_val_pred))
results['test_accuracy'].append(accuracy_score(test_df['sentiment'], y_test_pred))
results['test_f1_macro'].append(f1_score(test_df['sentiment'], y_test_pred, average='macro'))

## 2.2 PhoBERT Fine-Tuning

In [7]:
# ============================================================
# Load PhoBERT with a 3-class classification head
# ============================================================
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'vinai/phobert-base',
    num_labels=NUM_CLASSES
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Device: {device}")
print(f"Total parameters:     {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda
Total parameters:     135,000,579
Trainable parameters: 135,000,579


In [8]:
# ============================================================
# Optimizer and learning rate scheduler
# AdamW with linear warmup prevents early training instability
# ============================================================
from transformers import get_linear_schedule_with_warmup

EPOCHS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Epochs: {EPOCHS}, Total steps: {total_steps}, Warmup steps: {warmup_steps}")

Epochs: 4, Total steps: 1432, Warmup steps: 143


In [9]:
# ============================================================
# Training and evaluation helper functions
# Separated for clarity and reuse in the training loop
# ============================================================
from tqdm.auto import tqdm

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, accuracy, f1, np.array(all_preds), np.array(all_labels)

In [10]:
# ============================================================
# Training loop with early stopping on validation F1
# Best checkpoint saved to Google Drive for persistence
# ============================================================
CHECKPOINT_DIR = '/content/drive/MyDrive/phobert-sentiment-analysis/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_f1 = 0
patience = 2
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f"Train — Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   — Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        model.save_pretrained(CHECKPOINT_DIR)
        tokenizer.save_pretrained(CHECKPOINT_DIR)
        print(f">>> Best model saved! F1: {best_val_f1:.4f}")
    else:
        patience_counter += 1
        print(f">>> No improvement. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")


Epoch 1/4


Training:   0%|          | 0/358 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Train — Loss: 0.3940, Acc: 0.8622
Val   — Loss: 0.2203, Acc: 0.9406, F1: 0.8100
>>> Best model saved! F1: 0.8100

Epoch 2/4


Training:   0%|          | 0/358 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Train — Loss: 0.1253, Acc: 0.9656
Val   — Loss: 0.1962, Acc: 0.9558, F1: 0.8708
>>> Best model saved! F1: 0.8708

Epoch 4/4


Training:   0%|          | 0/358 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Train — Loss: 0.1000, Acc: 0.9720
Val   — Loss: 0.2033, Acc: 0.9551, F1: 0.8654
>>> No improvement. Patience: 1/2

Training complete. Best val F1: 0.8708
